## 4.Structured Output

we use JSON for structured output and also for data validation we use Pydantic.<br>
### Pydantic: 
A Python class for defining what your data should look like and validating it.

User<br>
 ↓<br>
"Give me candidate information for Vineetha"<br>
 ↓<br>
Gemini<br>
 ↓<br>
{<br>
  "name": "Vineetha",<br>
  "skills": ["Python", "SQL"],<br>
  "cgpa": 8.72,<br>
  "interest": "AI"<br>
}<br>
 ↓<br>
Pydantic<br>
 ↓<br>
Candidate object<br>
 ↓<br>
candidate.name<br>
candidate.skills<br>
candidate.cgpa

In [1]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b")

In [4]:
response = model.invoke(
    """
    Convert the following candidate information into a structured description:

    Name is Vineetha,Skills are Python, Java, SQL, FastAPI,CGPA is 8.72,Area of Interest is AI and Machine Learning

    Return the information clearly.
    """
)

print(response.content)

**Structured Candidate Profile**

```json
{
  "Name": "Vineetha",
  "Skills": [
    "Python",
    "Java",
    "SQL",
    "FastAPI"
  ],
  "CGPA": 8.72,
  "Area of Interest": "AI and Machine Learning"
}
```

This format presents the candidate’s key details in a clear, machine‑readable structure.


In [ ]:
# Creating pydantic schema
from pydantic import BaseModel

class Candidate(BaseModel):
    name:str
    skills:list[str]
    cgpa:float
    interest:str

In [6]:
print(Candidate.model_json_schema())

{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'skills': {'items': {'type': 'string'}, 'title': 'Skills', 'type': 'array'}, 'cgpa': {'title': 'Cgpa', 'type': 'number'}, 'interest': {'title': 'Interest', 'type': 'string'}}, 'required': ['name', 'skills', 'cgpa', 'interest'], 'title': 'Candidate', 'type': 'object'}


In [ ]:
# Creating a Pydantic object manually
candidate = Candidate(
    name="Vineetha",
    skills=["Python", "SQL"],
    cgpa=8.72,
    interest="AI"
)
print(candidate)
print(candidate.name)
print(candidate.skills)
print(candidate.cgpa)
print(candidate.interest)

name='Vineetha' skills=['Python', 'SQL'] cgpa=8.72 interest='AI'
Vineetha
['Python', 'SQL']
8.72
AI


### model.with_structured_output(Candidate):
"Groq, when you answer me, I don't want a random text response. I want the answer to follow this Candidate structure."

In [12]:
structured_model=model.with_structured_output(Candidate)
response = structured_model.invoke(
    """
    Convert the following candidate information into a structured description:
    Name is Vineetha,and my skills are Python,Java,SQL,,FastAPI,and i have 8.72 cgpa and my area of interest is AI and ml.
    return the information clealry
    """
    
)

print(response)

name='Vineetha' skills=['Python', 'Java', 'SQL', 'FastAPI'] cgpa=8.72 interest='AI and ml'


In [13]:
print(type(response))

<class '__main__.Candidate'>


In [15]:
print(response.name)
print(response.skills)
print(response.cgpa)
print(response.interest)

Vineetha
['Python', 'Java', 'SQL', 'FastAPI']
8.72
AI and ml


| Code                                      | What you get                         |
| ----------------------------------------- | ------------------------------------ |
| `model.invoke()`                          | `AIMessage`                          |
| `model.invoke()` + "give JSON"            | Usually JSON-looking **text**        |
| `Candidate(...)`                          | Manually created Pydantic object     |
| `model.with_structured_output(Candidate)` | LLM → **validated Candidate object** |


## Mini-project 🛠️
Resume Information Extractor

Input:

Resume text

Output:

{
  "name": "...",
  "skills": [],
  "education": [],
  "projects": [],
  "experience": []
}

This is actually useful for your AI portfolio.

In [19]:
from pypdf import PdfReader

pdf=PdfReader("JVineethaResume.pdf")
len(pdf.pages)

1

In [21]:
resume_text=""
for page in pdf.pages:
    resume_text+=page.extract_text() + "\n"

print(resume_text)

Vineetha Jakkilinki
/githubGitHub | /linkedinLinkedIn | /codeLeetCode | ✉ vineethajakkilinki@gmail.com | ♂¶obile+91 8019525680
Summary
Computer Science undergraduate with a strong foundation in Data Structures, Algorithms, Software Development, and Machine
Learning. Skilled in Python, Java, FastAPI, REST APIs, SQL, and Generative AI technologies, with hands-on experience building
backend services, AI-powered applications, semantic search systems, and LLM-based assistants. Experienced in developing and
deploying practical software solutions using APIs, databases, and modern AI frameworks.
Education
Vellore Institute of Technology
B.Tech in Computer Science Engineering (CGPA: 8.72) 2023 – 2027
Skills
Programming: Python, Java
Software Development: Data Structures Algorithms, Object-Oriented Programming, REST APIs
Frontend: HTML,CSS,React.js
Backend & APIs: FastAPI
Databases: MySQL
AI & Machine Learning: Scikit-learn, LangChain, LLMs , RAG, Vector Databases, Embeddings, Semantic Search, P

In [23]:
from pydantic import BaseModel

class Resume(BaseModel):
    name:str
    skills:list[str]
    education:list[str]
    projects:list[str]
    experience:list[str]
    certifications:list[str]

In [24]:
structured_model=model.with_structured_output(Resume)
res=structured_model.invoke(
    f"""
    Extract the following information from this resume:
    {resume_text}
    """
)

In [26]:
print(res)

name='Vineetha Jakkilinki' skills=['Programming: Python, Java', 'Software Development: Data Structures Algorithms, Object-Oriented Programming, REST APIs', 'Frontend: HTML,CSS,React.js', 'Backend & APIs: FastAPI', 'Databases: MySQL', 'AI & Machine Learning: Scikit-learn, LangChain, LLMs , RAG, Vector Databases, Embeddings, Semantic Search, Prompt Engineering', 'Data Science: NumPy, Pandas, Data Preprocessing, Data Cleaning, EDA, Feature Engineering, Model Evaluation', 'Cloud & Deployment: AWS, Hugging Face Spaces, Streamlit Cloud'] education=['Vellore Institute of Technology – B.Tech in Computer Science Engineering (CGPA: 8.72) 2023 – 2027'] projects=['AI-Powered Healthcare Assistant: Python, Scikit-learn, Random Forest, Groq API, Streamlit – Developed a machine learning-based application for disease prediction from user-provided symptoms; trained and evaluated Decision Tree, Random Forest, SVM, Gradient Boosting, and AdaBoost models, achieving 96% accuracy with Random Forest; integrat

In [27]:
print("Name:", res.name)
print("Skills:", res.skills)
print("Education:", res.education)
print("Projects:", res.projects)
print("Experience:", res.experience)

Name: Vineetha Jakkilinki
Skills: ['Programming: Python, Java', 'Software Development: Data Structures Algorithms, Object-Oriented Programming, REST APIs', 'Frontend: HTML,CSS,React.js', 'Backend & APIs: FastAPI', 'Databases: MySQL', 'AI & Machine Learning: Scikit-learn, LangChain, LLMs , RAG, Vector Databases, Embeddings, Semantic Search, Prompt Engineering', 'Data Science: NumPy, Pandas, Data Preprocessing, Data Cleaning, EDA, Feature Engineering, Model Evaluation', 'Cloud & Deployment: AWS, Hugging Face Spaces, Streamlit Cloud']
Education: ['Vellore Institute of Technology – B.Tech in Computer Science Engineering (CGPA: 8.72) 2023 – 2027']
Projects: ['AI-Powered Healthcare Assistant: Python, Scikit-learn, Random Forest, Groq API, Streamlit – Developed a machine learning-based application for disease prediction from user-provided symptoms; trained and evaluated Decision Tree, Random Forest, SVM, Gradient Boosting, and AdaBoost models, achieving 96% accuracy with Random Forest; integr